# EP2 — Agente Inteligente FTTH | Copia de revisión para GitHub

**Estado:** copia de desarrollo, todavía no consolidada como notebook de ejecución lineal.

Este archivo conserva la evolución desde el RAG de EP1 hasta el grafo V8 y las pruebas de persistencia, pero se eliminaron las salidas de ejecución para reducir tamaño y evitar publicar resultados accidentalmente. **No se modificó el código original de las celdas.**

**Antes de ejecutar o publicar:**
- Proporcionar el Excel de entrada anonimizado por separado; la primera celda depende de una ruta `/content/...xlsx`.
- Configurar `GOOGLE_API_KEY` en secretos de Colab; no escribir la clave en el notebook.
- Verificar el identificador y disponibilidad del modelo Gemini configurado en la celda correspondiente.
- Revisar las pruebas y distinguir las mediciones simuladas de los antecedentes reales.
- La memoria JSON en Drive restaura datos del estado, no los checkpoints completos.
- El flujo de herramientas y decisiones es principalmente determinista; no afirmar selección autónoma de herramientas por LLM.

**Pendiente:** consolidación en notebook reproducible, dependencias, dataset de ejemplo y pruebas sin llamadas externas.


In [ ]:
import pandas as pd

archivo = "/content/EP1_FTTH_Base_Trabajo_Inicial (2).xlsx"

df = pd.read_excel(
    archivo,
    sheet_name="FTTH_Base_Trabajo"
)

print("Base FTTH cargada correctamente")
print("Cantidad de registros:", len(df))
print("Cantidad de columnas:", len(df.columns))

df.head()

In [ ]:
print("COLUMNAS DISPONIBLES:")
print(df.columns.tolist())

print("\nTIPOS DE FALLA MÁS FRECUENTES:")
print(df["Motivo del corte"].value_counts().head(15))

In [ ]:
# Seleccionamos algunos casos con fallas útiles para nuestro prototipo
categorias_prototipo = [
    "Atenuación",
    "Filamento Atenuado",
    "Filamento cortado",
    "Caja CTO en mal estado"
]

df_prototipo = df[
    df["Motivo del corte"].isin(categorias_prototipo)
].copy()

print("Casos disponibles para el prototipo:", len(df_prototipo))

print("\nCantidad por categoría:")
print(df_prototipo["Motivo del corte"].value_counts())

In [ ]:
columnas_revision = [
    "Caso_ID",
    "Motivo del corte",
    "Distancia",
    "Acción Reparación",
    "Materiales Ocupados"
]

for categoria in categorias_prototipo:
    print("\n" + "=" * 80)
    print("CATEGORÍA:", categoria)
    print("=" * 80)

    ejemplo = df_prototipo[
        df_prototipo["Motivo del corte"] == categoria
    ][columnas_revision].head(3)

    display(ejemplo)

In [ ]:
caso = df[df["Caso_ID"] == "FTTH-0010"].iloc[0]

print("CASO HISTÓRICO FTTH-0010")
print("=" * 60)

print("Tipo de red:", caso["Tipo Red"])
print("Motivo registrado:", caso["Motivo del corte"])
print("Distancia:", caso["Distancia"])
print("Comuna:", caso["Comuna"])
print("ETR:", caso["ETR (horas)"], "horas")

print("\nACCIÓN DE REPARACIÓN:")
print(caso["Acción Reparación"])

print("\nMATERIALES OCUPADOS:")
print(caso["Materiales Ocupados"])

In [ ]:
def crear_documento_ftth(fila):
    texto = f"""
Caso histórico: {fila['Caso_ID']}
Tipo de red: {fila['Tipo Red']}
Motivo registrado: {fila['Motivo del corte']}
Distancia aproximada: {fila['Distancia']} metros
Acción de reparación realizada: {fila['Acción Reparación']}
Uso de materiales: {fila['Materiales Ocupados']}
"""
    return texto.strip()


df_prototipo["Documento_RAG"] = df_prototipo.apply(
    crear_documento_ftth,
    axis=1
)

print("DOCUMENTO PREPARADO PARA RAG")
print("=" * 60)
print(df_prototipo.iloc[0]["Documento_RAG"])

In [ ]:
!pip install -qU langchain-google-genai langchain-chroma

In [ ]:
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)
from langchain_chroma import Chroma

print("✅ Librerías de LangChain, Gemini y Chroma importadas correctamente")

In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

print("✅ Credenciales de Google Gemini configuradas correctamente")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

print("Importación correcta")
print("GoogleGenerativeAIEmbeddings: OK")
print("Chroma: OK")

In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Modelo de embeddings configurado correctamente")

In [ ]:
texto_prueba = """
Incidencia FTTH con atenuación óptica.
Se detecta degradación en el enlace y se requiere
revisión de empalmes y elementos de la ruta.
"""

vector_prueba = embeddings.embed_query(texto_prueba)

print("Embedding generado correctamente")
print("Cantidad de dimensiones:", len(vector_prueba))
print("Primeros 10 valores del vector:")
print(vector_prueba[:10])

In [ ]:
import numpy as np
from numpy.linalg import norm

texto_a = "Incidencia FTTH con atenuación óptica y degradación del nivel de señal."

texto_b = "Enlace de fibra presenta pérdida de potencia óptica y se requiere revisar empalmes."

texto_c = "Caja CTO dañada físicamente y requiere reparación de su estructura."

vector_a = embeddings.embed_query(texto_a)
vector_b = embeddings.embed_query(texto_b)
vector_c = embeddings.embed_query(texto_c)

def similitud_coseno(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sim_ab = similitud_coseno(vector_a, vector_b)
sim_ac = similitud_coseno(vector_a, vector_c)

print("SIMILITUD SEMÁNTICA")
print("=" * 50)
print(f"Atenuación vs pérdida óptica : {sim_ab:.4f}")
print(f"Atenuación vs CTO dañada     : {sim_ac:.4f}")

In [ ]:
muestras = []

for categoria in categorias_prototipo:
    casos_categoria = df_prototipo[
        df_prototipo["Motivo del corte"] == categoria
    ].head(20)

    muestras.append(casos_categoria)

df_rag = pd.concat(muestras, ignore_index=True)

print("BASE DE CONOCIMIENTO INICIAL")
print("=" * 50)
print("Total de documentos:", len(df_rag))

print("\nDistribución:")
print(df_rag["Motivo del corte"].value_counts())

In [ ]:
from langchain_chroma import Chroma

textos = df_rag["Documento_RAG"].tolist()

metadatos = []

for _, fila in df_rag.iterrows():
    metadatos.append({
        "caso_id": fila["Caso_ID"],
        "motivo": fila["Motivo del corte"]
    })

print("Documentos preparados:", len(textos))
print("Metadatos preparados:", len(metadatos))

In [ ]:
from langchain_chroma import Chroma

base_vectorial = Chroma.from_texts(
    texts=textos,
    embedding=embeddings,
    metadatas=metadatos,
    collection_name="incidencias_ftth"
)

print("BASE VECTORIAL CREADA CORRECTAMENTE")
print("=" * 50)
print("Documentos indexados:", base_vectorial._collection.count())

In [ ]:
consulta = """
Enlace FTTH presenta degradación de potencia óptica.
Se sospecha una atenuación en la fibra y se requiere
revisar empalmes o mufas de la ruta.
"""

resultados = base_vectorial.similarity_search(
    consulta,
    k=3
)

print("CONSULTA DEL INCIDENTE")
print("=" * 60)
print(consulta)

print("\nCASOS HISTÓRICOS RECUPERADOS POR EL RAG")
print("=" * 60)

for i, documento in enumerate(resultados, start=1):
    print(f"\nRESULTADO {i}")
    print("Caso:", documento.metadata["caso_id"])
    print("Motivo:", documento.metadata["motivo"])
    print("-" * 40)
    print(documento.page_content)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

print("LLM configurado correctamente")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_ftth = ChatPromptTemplate.from_template("""
Eres un asistente técnico de apoyo para el análisis de incidencias
en redes FTTH.

Debes analizar la nueva incidencia utilizando ÚNICAMENTE los casos
históricos recuperados que aparecen en el contexto.

REGLAS:
1. No inventes mediciones, causas, reparaciones ni antecedentes.
2. No afirmes que una causa está confirmada si los datos no lo permiten.
3. Identifica patrones observados en los casos históricos recuperados.
4. Propón verificaciones técnicas basadas solamente en esos antecedentes.
5. Indica los Caso_ID utilizados como respaldo.
6. La decisión final corresponde al personal técnico en terreno.

NUEVA INCIDENCIA:
{incidencia}

CASOS HISTÓRICOS RECUPERADOS:
{contexto}

Entrega la respuesta utilizando esta estructura:

ANÁLISIS:
Breve análisis de la incidencia.

CASOS SIMILARES:
Indica los Caso_ID recuperados y por qué son relevantes.

POSIBLE ORIENTACIÓN:
Indica qué condición podría estar relacionada, sin presentarla como
diagnóstico confirmado.

VERIFICACIONES SUGERIDAS:
Enumera las verificaciones que se podrían realizar según los antecedentes.

TRAZABILIDAD:
Indica explícitamente qué casos históricos respaldan la respuesta.
""")

print("Prompt FTTH creado correctamente")

In [ ]:
# Nueva incidencia para probar el sistema RAG
nueva_incidencia = """
Incidencia en red FTTH.
Se detecta degradación de potencia óptica en un cliente.
La potencia medida se encuentra aproximadamente en -25 dBm.
Se requiere determinar qué verificaciones realizar antes de efectuar
una reparación.
"""

# 1. Recuperar los 3 casos históricos más similares
casos_recuperados = base_vectorial.similarity_search(
    nueva_incidencia,
    k=3
)

# 2. Construir el contexto para el LLM
contexto = "\n\n".join(
    [
        f"CASO {doc.metadata.get('caso_id')}\n{doc.page_content}"
        for doc in casos_recuperados
    ]
)

print("INCIDENCIA NUEVA")
print("=" * 60)
print(nueva_incidencia)

print("\nCASOS RECUPERADOS POR EL RAG")
print("=" * 60)

for i, doc in enumerate(casos_recuperados, 1):
    print(f"\nResultado {i}")
    print("Caso:", doc.metadata.get("caso_id"))
    print("Motivo:", doc.metadata.get("motivo"))

In [ ]:
# Construimos el prompt final utilizando la incidencia
# y los casos recuperados por el RAG

prompt_final = prompt_ftth.format(
    incidencia=nueva_incidencia,
    contexto=contexto
)

# Enviamos la información recuperada al modelo Gemini
respuesta = llm.invoke(prompt_final)

print("RESPUESTA DEL ASISTENTE RAG FTTH")
print("=" * 60)
print(respuesta.content)

In [ ]:
def obtener_texto_respuesta(respuesta):
    contenido = respuesta.content

    if isinstance(contenido, str):
        return contenido

    if isinstance(contenido, list):
        partes = []

        for elemento in contenido:
            if isinstance(elemento, dict):
                texto = elemento.get("text", "")
                if texto:
                    partes.append(texto)
            else:
                partes.append(str(elemento))

        return "\n".join(partes)

    return str(contenido)


respuesta_limpia = obtener_texto_respuesta(respuesta)

print("RESPUESTA FINAL DEL ASISTENTE RAG FTTH")
print("=" * 70)
print(respuesta_limpia)

In [ ]:
nueva_incidencia_2 = """
Incidencia en red FTTH.
Se detecta daño físico en una caja CTO.
La caja presenta deterioro y requiere revisión del estado
de los filamentos, pigtails y conexiones internas.
Se necesita determinar qué verificaciones realizar antes
de efectuar una reparación.
"""

# Recuperamos 3 casos históricos similares
casos_recuperados_2 = base_vectorial.similarity_search(
    nueva_incidencia_2,
    k=3
)

# Construimos el contexto
contexto_2 = "\n\n".join(
    [
        f"CASO {doc.metadata.get('caso_id')}\n{doc.page_content}"
        for doc in casos_recuperados_2
    ]
)

print("SEGUNDA INCIDENCIA DE PRUEBA")
print("=" * 60)
print(nueva_incidencia_2)

print("\nCASOS RECUPERADOS POR EL RAG")
print("=" * 60)

for i, doc in enumerate(casos_recuperados_2, 1):
    print(f"\nResultado {i}")
    print("Caso:", doc.metadata.get("caso_id"))
    print("Motivo:", doc.metadata.get("motivo"))

In [ ]:
prompt_final_2 = prompt_ftth.format(
    incidencia=nueva_incidencia_2,
    contexto=contexto_2
)

respuesta_2 = llm.invoke(prompt_final_2)

respuesta_limpia_2 = obtener_texto_respuesta(respuesta_2)

print("RESPUESTA FINAL - PRUEBA CTO")
print("=" * 70)
print(respuesta_limpia_2)

## Fuente externa de conocimiento técnico FTTH / GPON

In [ ]:
# Fuente externa oficial para complementar la base RAG
# Referencia: ITU-T G.984.1 - Gigabit-capable passive optical networks (GPON)

fuente_externa_gpon = """
FUENTE EXTERNA: ITU-T G.984.1

Las redes GPON corresponden a redes ópticas pasivas de acceso.

Una arquitectura GPON está compuesta generalmente por:
- OLT (Optical Line Termination).
- ONU u ONT (Optical Network Unit / Optical Network Termination).
- ODN (Optical Distribution Network), que interconecta la OLT con las ONU/ONT.

La red de distribución óptica es pasiva y permite una relación
de uno a muchos entre la OLT y las ONU/ONT.

En el análisis de una red óptica deben considerarse las características
de la infraestructura óptica y los elementos que forman parte de la ODN.

Referencia:
ITU-T Recommendation G.984.1
Gigabit-capable passive optical networks (GPON): General characteristics.
"""

print("FUENTE EXTERNA CARGADA")
print("=" * 60)
print(fuente_externa_gpon)

In [ ]:
# Incorporamos la fuente externa a la base vectorial

base_vectorial.add_texts(
    texts=[fuente_externa_gpon],
    metadatas=[
        {
            "caso_id": "ITU-G984.1",
            "motivo": "Documentación técnica GPON",
            "tipo_fuente": "fuente_externa"
        }
    ]
)

print("FUENTE EXTERNA INCORPORADA AL RAG")
print("=" * 60)
print("Documento externo: ITU-T G.984.1")
print("Total de documentos indexados:", base_vectorial._collection.count())

In [ ]:
consulta_externa = """
¿Cuáles son los principales elementos que componen
una arquitectura de red GPON?
"""

resultados_externos = base_vectorial.similarity_search(
    consulta_externa,
    k=3
)

print("PRUEBA DE RECUPERACIÓN DE FUENTE EXTERNA")
print("=" * 60)

for i, doc in enumerate(resultados_externos, 1):
    print(f"\nResultado {i}")
    print("ID:", doc.metadata.get("caso_id"))
    print("Motivo:", doc.metadata.get("motivo"))
    print("Tipo de fuente:", doc.metadata.get("tipo_fuente", "fuente_interna"))

In [ ]:
consulta_combinada = """
Incidencia FTTH con degradación de potencia óptica.
Se requiere orientar la revisión considerando tanto
casos históricos de reparación como elementos generales
de la arquitectura GPON involucrados en la red.
"""

resultados_combinados = base_vectorial.similarity_search(
    consulta_combinada,
    k=5
)

print("PRUEBA DE RECUPERACIÓN COMBINADA")
print("=" * 60)

for i, doc in enumerate(resultados_combinados, 1):
    print(f"\nResultado {i}")
    print("ID:", doc.metadata.get("caso_id"))
    print("Motivo:", doc.metadata.get("motivo"))
    print("Tipo de fuente:", doc.metadata.get("tipo_fuente", "fuente_interna"))

In [ ]:
muestra_metadata = base_vectorial._collection.get(limit=3)

print("METADATOS REALES GUARDADOS EN CHROMA")
print("=" * 60)

for metadata in muestra_metadata["metadatas"]:
    print(metadata)

In [ ]:
# Agregamos el metadato "tipo_fuente" a los casos históricos internos

datos_chroma = base_vectorial._collection.get()

ids_internos = []
metadatos_actualizados = []

for doc_id, metadata in zip(
    datos_chroma["ids"],
    datos_chroma["metadatas"]
):
    # Identificamos los casos históricos por su Caso_ID FTTH
    if metadata.get("caso_id", "").startswith("FTTH-"):

        nuevo_metadata = metadata.copy()
        nuevo_metadata["tipo_fuente"] = "fuente_interna"

        ids_internos.append(doc_id)
        metadatos_actualizados.append(nuevo_metadata)

# Actualizamos solamente los metadatos
base_vectorial._collection.update(
    ids=ids_internos,
    metadatas=metadatos_actualizados
)

print("METADATOS ACTUALIZADOS CORRECTAMENTE")
print("=" * 60)
print("Casos históricos actualizados:", len(ids_internos))

In [ ]:
consulta_mixta = """
Incidencia FTTH con degradación de potencia óptica.
Se requiere orientar técnicamente la revisión de la red,
considerando antecedentes históricos y documentación técnica GPON.
"""

# Recuperación desde casos históricos internos
resultados_internos = base_vectorial.similarity_search(
    consulta_mixta,
    k=3,
    filter={"tipo_fuente": "fuente_interna"}
)

# Recuperación desde documentación técnica externa
resultados_externos = base_vectorial.similarity_search(
    consulta_mixta,
    k=1,
    filter={"tipo_fuente": "fuente_externa"}
)

print("RECUPERACIÓN RAG: FUENTES INTERNAS + EXTERNAS")
print("=" * 65)

print("\nFUENTES INTERNAS RECUPERADAS:")
for doc in resultados_internos:
    print(
        "-",
        doc.metadata.get("caso_id"),
        "|",
        doc.metadata.get("motivo"),
        "|",
        doc.metadata.get("tipo_fuente")
    )

print("\nFUENTE EXTERNA RECUPERADA:")
for doc in resultados_externos:
    print(
        "-",
        doc.metadata.get("caso_id"),
        "|",
        doc.metadata.get("motivo"),
        "|",
        doc.metadata.get("tipo_fuente")
    )

In [ ]:
# Construimos un contexto combinado: fuentes internas + fuente externa

contexto_interno = "\n\n".join(
    [
        f"FUENTE INTERNA - CASO {doc.metadata.get('caso_id')}\n{doc.page_content}"
        for doc in resultados_internos
    ]
)

contexto_externo = "\n\n".join(
    [
        f"FUENTE EXTERNA - {doc.metadata.get('caso_id')}\n{doc.page_content}"
        for doc in resultados_externos
    ]
)

contexto_combinado = contexto_interno + "\n\n" + contexto_externo

print("CONTEXTO COMBINADO PREPARADO")
print("=" * 65)
print("Fuentes internas:", len(resultados_internos))
print("Fuentes externas:", len(resultados_externos))
print("Total de documentos utilizados:",
      len(resultados_internos) + len(resultados_externos))

In [ ]:
prompt_combinado = f"""
Eres un asistente de apoyo técnico para diagnóstico de incidencias
en redes FTTH.

Debes responder utilizando EXCLUSIVAMENTE la información contenida
en las fuentes recuperadas.

REGLAS:
1. No inventes mediciones, causas, reparaciones ni antecedentes.
2. Diferencia claramente los antecedentes históricos internos
   de la documentación técnica externa.
3. No presentes una posible causa como diagnóstico confirmado.
4. Propón verificaciones basadas solamente en las fuentes recuperadas.
5. Indica los Caso_ID utilizados como respaldo.
6. Si utilizas información externa, identifica la referencia ITU-G984.1.
7. La decisión final corresponde al personal técnico en terreno.

INCIDENCIA:
{consulta_mixta}

CONTEXTO RECUPERADO:
{contexto_combinado}

Entrega la respuesta utilizando esta estructura:

ANÁLISIS:
Breve análisis de la incidencia.

ANTECEDENTES INTERNOS:
Indica los casos históricos relevantes y qué aportan.

FUENTE EXTERNA:
Indica qué información aporta la documentación técnica externa.

POSIBLE ORIENTACIÓN:
Indica condiciones que podrían estar relacionadas, sin confirmarlas
como diagnóstico.

VERIFICACIONES SUGERIDAS:
Enumera verificaciones basadas en los antecedentes recuperados.

TRAZABILIDAD:
Identifica claramente las fuentes internas y externas utilizadas.
"""

respuesta_combinada = llm.invoke(prompt_combinado)

respuesta_combinada_limpia = obtener_texto_respuesta(
    respuesta_combinada
)

print("RESPUESTA FINAL RAG - FUENTES INTERNAS + EXTERNAS")
print("=" * 70)
print(respuesta_combinada_limpia)

In [ ]:
import matplotlib.pyplot as plt

# Contar los casos de cada categoría
cantidad_casos = (
    df_prototipo["Motivo del corte"]
    .value_counts()
    .sort_values(ascending=False)
)

# Mostrar los datos utilizados
print("CASOS FTTH POR CATEGORÍA")
print("=" * 40)
print(cantidad_casos)

# Crear gráfico
plt.figure(figsize=(10, 5))

cantidad_casos.plot(kind="bar")

plt.title("Casos FTTH disponibles para el prototipo")
plt.xlabel("Motivo del corte")
plt.ylabel("Cantidad de casos")

plt.xticks(rotation=25, ha="right")
plt.tight_layout()

plt.show()

# Evaluación Parcial 2 — Agente Inteligente para Diagnóstico FTTH

**Asignatura:** Ingeniería de Soluciones con Inteligencia Artificial (ISY0101)  
**Modalidad:** Trabajo individual  
**Entorno de desarrollo:** Google Colab / Python

## 1. Introducción

Esta evaluación corresponde a la continuación del prototipo desarrollado en la Evaluación Parcial 1, donde se implementó un asistente basado en modelos de lenguaje (LLM) y recuperación aumentada por generación (RAG) para apoyar el diagnóstico de incidencias en redes de fibra óptica FTTH.

La solución anterior permitió consultar antecedentes históricos de reparaciones y documentación técnica externa mediante embeddings, una base vectorial Chroma y Google Gemini.

En esta segunda etapa se busca ampliar las capacidades del prototipo mediante una arquitectura de agentes inteligentes, incorporando herramientas, memoria, planificación y decisiones condicionales para apoyar el análisis de incidencias.

## 2. Objetivo de la implementación

Desarrollar una evolución funcional del asistente RAG FTTH mediante LangGraph, integrando un agente supervisor capaz de analizar la información disponible, seleccionar herramientas, mantener el contexto de una incidencia y organizar las verificaciones sugeridas.

La solución continuará funcionando como herramienta de apoyo, manteniendo la validación y decisión final en el personal técnico.

## 3. Alcance inicial

El desarrollo se realizará progresivamente, reutilizando el sistema RAG implementado en la EP1 e incorporando nuevas capacidades de agente inteligente.

Se considerarán los siguientes componentes:

- Estado compartido para registrar los antecedentes de cada incidencia.
- Nodo supervisor para evaluar la información disponible.
- Herramientas para consultar antecedentes históricos y documentación técnica.
- Memoria para conservar el contexto de las interacciones.
- Planificación y decisiones condicionales según los antecedentes disponibles.
- Pruebas funcionales con incidencias simuladas y resultados trazables.

Las funcionalidades serán implementadas y verificadas durante el desarrollo de la evaluación.

---

## 4. Implementación del agente inteligente FTTH

### 4.1 Preparación del entorno

Se incorpora LangGraph como framework para construir el flujo del agente mediante estados, nodos y conexiones.

### 4.2 Estado compartido

Se define `EstadoFTTH` para mantener la información de la incidencia, incluyendo identificación, infraestructura, mediciones, antecedentes recuperados, planificación y respuesta.

### 4.3 Nodo supervisor

Se define un primer nodo supervisor que evalúa la información recibida y determina el siguiente paso mediante reglas condicionales. Esta implementación inicial será ampliada posteriormente con herramientas y recuperación de contexto.

In [ ]:
# EP2 - Instalación de LangGraph

!pip install -qU langgraph

print("Instalación de LangGraph finalizada")

In [ ]:
# EP2 - Verificación de LangGraph

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

print("LangGraph importado correctamente")
print("StateGraph: OK")
print("Memoria: OK")

In [ ]:
# EP2 - Definición del estado compartido del agente FTTH

from typing import TypedDict, List

class EstadoFTTH(TypedDict):
    ticket: str
    tipo_red: str
    descripcion: str
    nodo: str
    rack: str
    odf: str
    posicion: str
    distancia_otdr: float | None
    antecedentes: List[str]
    plan_revision: List[str]
    siguiente_paso: str
    respuesta_final: str

print("EstadoFTTH definido correctamente")
print("Campos disponibles:")

for campo in EstadoFTTH.__annotations__:
    print("-", campo)

In [ ]:
# EP2 - Primer nodo supervisor FTTH

def nodo_supervisor(estado: EstadoFTTH):

    print("=== SUPERVISOR FTTH ===")
    print("Ticket:", estado["ticket"])

    # Verificar si existe una descripción de la incidencia
    if not estado["descripcion"].strip():
        siguiente_paso = "solicitar_informacion"

    # Verificar si existen mediciones OTDR
    elif estado["distancia_otdr"] is None:
        siguiente_paso = "solicitar_medicion"

    # Si existen antecedentes suficientes, consultar RAG
    else:
        siguiente_paso = "consultar_rag"

    print("Decisión del supervisor:", siguiente_paso)

    return {"siguiente_paso": siguiente_paso}


print("Nodo supervisor FTTH creado correctamente")

In [ ]:
# EP2 - Pruebas del nodo supervisor FTTH

# Caso 1: Incidencia sin descripción
caso_1 = {
    "ticket": "SIM-001",
    "descripcion": "",
    "distancia_otdr": None
}

# Caso 2: Incidencia con descripción, pero sin medición OTDR
caso_2 = {
    "ticket": "SIM-002",
    "descripcion": "Cliente FTTH presenta pérdida de señal óptica",
    "distancia_otdr": None
}

# Caso 3: Incidencia con descripción y medición OTDR
caso_3 = {
    "ticket": "SIM-003",
    "descripcion": "Cliente FTTH presenta degradación de señal",
    "distancia_otdr": 2450.0
}

print("PRUEBA 1")
resultado_1 = nodo_supervisor(caso_1)

print("\nPRUEBA 2")
resultado_2 = nodo_supervisor(caso_2)

print("\nPRUEBA 3")
resultado_3 = nodo_supervisor(caso_3)

In [ ]:
# EP2 - Primer grafo funcional FTTH con LangGraph

def nodo_solicitar_informacion(estado: EstadoFTTH):
    return {
        "respuesta_final": "Falta la descripción de la incidencia."
    }

def nodo_solicitar_medicion(estado: EstadoFTTH):
    return {
        "respuesta_final": (
            "No se dispone de distancia OTDR. "
            "Se solicita medición para continuar este flujo inicial."
        )
    }

def nodo_consultar_rag(estado: EstadoFTTH):
    return {
        "respuesta_final": (
            "La incidencia cuenta con descripción y distancia OTDR. "
            "Está preparada para consultar el sistema RAG."
        )
    }

# Crear el grafo
builder = StateGraph(EstadoFTTH)

# Registrar nodos
builder.add_node("supervisor", nodo_supervisor)
builder.add_node("solicitar_informacion", nodo_solicitar_informacion)
builder.add_node("solicitar_medicion", nodo_solicitar_medicion)
builder.add_node("consultar_rag", nodo_consultar_rag)

# Definir el inicio
builder.add_edge(START, "supervisor")

# El supervisor decide qué nodo ejecutar
builder.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "solicitar_medicion": "solicitar_medicion",
        "consultar_rag": "consultar_rag"
    }
)

# Finalizar después del nodo seleccionado
builder.add_edge("solicitar_informacion", END)
builder.add_edge("solicitar_medicion", END)
builder.add_edge("consultar_rag", END)

grafo_ftth = builder.compile()

print("Grafo FTTH compilado correctamente")

In [ ]:
# EP2 - Pruebas funcionales del grafo FTTH

casos_prueba = [
    {
        "ticket": "SIM-001",
        "descripcion": "",
        "distancia_otdr": None
    },
    {
        "ticket": "SIM-002",
        "descripcion": "Cliente FTTH presenta pérdida de señal óptica",
        "distancia_otdr": None
    },
    {
        "ticket": "SIM-003",
        "descripcion": "Cliente FTTH presenta degradación de señal",
        "distancia_otdr": 2450.0
    }
]

for caso in casos_prueba:

    print("\n" + "=" * 55)
    print("EJECUTANDO:", caso["ticket"])

    resultado = grafo_ftth.invoke(caso)

    print("Ruta seleccionada:", resultado["siguiente_paso"])
    print("Respuesta:", resultado["respuesta_final"])

### 4.4 Pruebas iniciales del grafo FTTH

Se realizaron tres pruebas funcionales con incidencias simuladas para comprobar el comportamiento del nodo supervisor y las rutas condicionales implementadas mediante LangGraph.

| Caso | Condición evaluada | Ruta ejecutada | Resultado |
|---|---|---|---|
| SIM-001 | Incidencia sin descripción | `solicitar_informacion` | Correcto |
| SIM-002 | Incidencia con descripción, sin distancia OTDR | `solicitar_medicion` | Correcto |
| SIM-003 | Incidencia con descripción y distancia OTDR | `consultar_rag` | Correcto |

Las pruebas permitieron verificar que el grafo ejecuta diferentes nodos según la información disponible.

En esta versión inicial, las decisiones se realizan mediante reglas condicionales de Python. El nodo `consultar_rag` todavía entrega una respuesta simulada y será conectado posteriormente al sistema RAG desarrollado en la EP1.

**Estado:** Primera versión del grafo funcional validada mediante tres escenarios simulados.

In [ ]:
# EP2 - Verificación de componentes RAG disponibles

print("=== COMPONENTES DISPONIBLES ===")

for nombre in [
    "embeddings",
    "vectorstore",
    "vector_store",
    "db",
    "chroma_db",
    "llm",
    "documentos",
    "documentos_rag"
]:
    if nombre in globals():
        objeto = globals()[nombre]
        print(f"OK - {nombre}: {type(objeto).__name__}")
    else:
        print(f"No encontrado - {nombre}")

In [ ]:
# EP2 - Localizar la base vectorial Chroma existente

from langchain_chroma import Chroma

print("=== BÚSQUEDA DE BASES CHROMA ===")

bases_encontradas = []

for nombre, objeto in list(globals().items()):
    if isinstance(objeto, Chroma):
        bases_encontradas.append(nombre)
        print("Base encontrada:", nombre)

        try:
            print("Documentos almacenados:", objeto._collection.count())
        except Exception as error:
            print("No se pudo consultar el total:", error)

if not bases_encontradas:
    print("No se encontraron objetos Chroma activos.")

In [ ]:
# EP2 - Herramienta de consulta RAG FTTH

def herramienta_consulta_rag(descripcion, cantidad=3):

    print("=== CONSULTA RAG FTTH ===")
    print("Incidencia:", descripcion)

    # Recuperar antecedentes desde Chroma
    documentos = base_vectorial.similarity_search(
        descripcion,
        k=cantidad
    )

    resultados = []

    for documento in documentos:

        caso_id = documento.metadata.get(
            "caso_id", "Sin identificador"
        )

        tipo_fuente = documento.metadata.get(
            "tipo_fuente", "No especificada"
        )

        resultados.append({
            "caso_id": caso_id,
            "tipo_fuente": tipo_fuente,
            "contenido": documento.page_content
        })

    print("Antecedentes recuperados:", len(resultados))

    return resultados


print("Herramienta de consulta RAG creada correctamente")

In [ ]:
# EP2 - Primera prueba de consulta RAG

consulta_prueba = (
    "Cliente FTTH presenta degradación de señal óptica "
    "y posible atenuación en la ruta."
)

antecedentes_prueba = herramienta_consulta_rag(
    consulta_prueba,
    cantidad=3
)

for antecedente in antecedentes_prueba:

    print("\n----------------------------")
    print("Caso:", antecedente["caso_id"])
    print("Fuente:", antecedente["tipo_fuente"])
    print("Contenido:")
    print(antecedente["contenido"][:500])

In [ ]:
# EP2 - Conectar el nodo RAG real con LangGraph

def nodo_consultar_rag(estado: EstadoFTTH):

    descripcion = estado["descripcion"]

    print("=== NODO RAG REAL ===")

    antecedentes = herramienta_consulta_rag(
        descripcion,
        cantidad=3
    )

    textos = []

    for caso in antecedentes:
        textos.append(
            f"Fuente: {caso['tipo_fuente']} | "
            f"ID: {caso['caso_id']}\n"
            f"{caso['contenido']}"
        )

    identificadores = [
        caso["caso_id"] for caso in antecedentes
    ]

    return {
        "antecedentes": textos,
        "respuesta_final": (
            "Consulta RAG realizada correctamente. "
            "Antecedentes recuperados: "
            + ", ".join(identificadores)
            if identificadores
            else "No se encontraron antecedentes."
        )
    }


# Reconstruir el grafo con el nodo RAG actualizado

builder_ep2 = StateGraph(EstadoFTTH)

builder_ep2.add_node("supervisor", nodo_supervisor)
builder_ep2.add_node(
    "solicitar_informacion",
    nodo_solicitar_informacion
)
builder_ep2.add_node(
    "solicitar_medicion",
    nodo_solicitar_medicion
)
builder_ep2.add_node("consultar_rag", nodo_consultar_rag)

builder_ep2.add_edge(START, "supervisor")

builder_ep2.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "solicitar_medicion": "solicitar_medicion",
        "consultar_rag": "consultar_rag"
    }
)

for nodo in [
    "solicitar_informacion",
    "solicitar_medicion",
    "consultar_rag"
]:
    builder_ep2.add_edge(nodo, END)

grafo_ftth_ep2 = builder_ep2.compile()

print("Grafo EP2 conectado con RAG real correctamente")

In [ ]:
# EP2 - Prueba integrada LangGraph + RAG real

incidencia_prueba = {
    "ticket": "SIM-004",
    "tipo_red": "FTTH",
    "descripcion": (
        "Cliente FTTH presenta degradación de señal óptica "
        "y posible atenuación en la ruta."
    ),
    "nodo": "NODO-SIMULADO",
    "rack": "1",
    "odf": "2",
    "posicion": "10",
    "distancia_otdr": 2450.0,
    "antecedentes": [],
    "plan_revision": [],
    "siguiente_paso": "",
    "respuesta_final": ""
}

resultado_ep2 = grafo_ftth_ep2.invoke(incidencia_prueba)

print("\n=== RESULTADO DEL AGENTE FTTH ===")
print("Ticket:", resultado_ep2["ticket"])
print("Ruta seleccionada:", resultado_ep2["siguiente_paso"])
print("Respuesta:", resultado_ep2["respuesta_final"])

print("\n=== ANTECEDENTES RECUPERADOS ===")
print("Cantidad:", len(resultado_ep2["antecedentes"]))

for numero, antecedente in enumerate(
    resultado_ep2["antecedentes"], start=1
):
    print(f"\nAntecedente {numero}")
    print(antecedente[:350])

### 4.5 Integración funcional entre LangGraph y RAG

Se realizó una prueba de integración utilizando la incidencia simulada SIM-004, relacionada con degradación de señal óptica y posible atenuación en una red FTTH.

El grafo ejecutó el nodo supervisor, seleccionó la ruta `consultar_rag` y realizó una búsqueda semántica en la base vectorial Chroma implementada durante la EP1.

**Resultados obtenidos:**

- Ruta seleccionada: `consultar_rag`.
- Antecedentes recuperados: 3.
- Casos identificados: FTTH-0437, FTTH-0134 y FTTH-0356.
- Fuentes recuperadas: registros históricos internos.
- Estado de la prueba: correcto.

Los antecedentes fueron incorporados al estado compartido del grafo, permitiendo su utilización en etapas posteriores.

Esta prueba confirma el funcionamiento de la recuperación semántica integrada al flujo de LangGraph. La generación de orientaciones técnicas mediante Gemini, la memoria entre interacciones y la planificación de verificaciones se incorporarán en etapas posteriores.

In [ ]:
# EP2 - Supervisor mejorado para consultas sin OTDR

def nodo_supervisor(estado: EstadoFTTH):

    print("=== SUPERVISOR FTTH V2 ===")
    print("Ticket:", estado.get("ticket", "Sin ticket"))

    descripcion = (estado.get("descripcion") or "").strip()
    distancia = estado.get("distancia_otdr")

    if not descripcion:
        siguiente_paso = "solicitar_informacion"
    else:
        siguiente_paso = "consultar_rag"

    if distancia is None:
        print("Observación: No existe medición OTDR disponible.")
    else:
        print("Distancia OTDR disponible:", distancia, "metros")

    print("Decisión del supervisor:", siguiente_paso)

    return {"siguiente_paso": siguiente_paso}


# Reconstruir el grafo para incorporar el supervisor V2
builder_v2 = StateGraph(EstadoFTTH)

builder_v2.add_node("supervisor", nodo_supervisor)
builder_v2.add_node("solicitar_informacion", nodo_solicitar_informacion)
builder_v2.add_node("solicitar_medicion", nodo_solicitar_medicion)
builder_v2.add_node("consultar_rag", nodo_consultar_rag)

builder_v2.add_edge(START, "supervisor")

builder_v2.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "solicitar_medicion": "solicitar_medicion",
        "consultar_rag": "consultar_rag"
    }
)

builder_v2.add_edge("solicitar_informacion", END)
builder_v2.add_edge("solicitar_medicion", END)
builder_v2.add_edge("consultar_rag", END)

grafo_ftth_v2 = builder_v2.compile()

print("Supervisor V2 y grafo actualizado correctamente")

In [ ]:
# EP2 - Prueba del supervisor V2 sin medición OTDR

incidencia_sin_otdr = {
    "ticket": "SIM-005",
    "tipo_red": "FTTH",
    "descripcion": (
        "Clientes FTTH sin servicio. "
        "Se informa posible daño físico en caja CTO."
    ),
    "nodo": "NODO-SIMULADO",
    "rack": "",
    "odf": "",
    "posicion": "",
    "distancia_otdr": None,
    "antecedentes": [],
    "plan_revision": [],
    "siguiente_paso": "",
    "respuesta_final": ""
}

resultado_v2 = grafo_ftth_v2.invoke(incidencia_sin_otdr)

print("\n=== RESULTADO SUPERVISOR V2 ===")
print("Ticket:", resultado_v2["ticket"])
print("Ruta:", resultado_v2["siguiente_paso"])
print("Respuesta:", resultado_v2["respuesta_final"])

print("\n=== ANTECEDENTES HISTÓRICOS ===")
print("Cantidad:", len(resultado_v2["antecedentes"]))

for numero, antecedente in enumerate(
    resultado_v2["antecedentes"], start=1
):
    print(f"\nAntecedente {numero}")
    print(antecedente[:350])

# Validaciones automáticas
assert resultado_v2["siguiente_paso"] == "consultar_rag"
assert len(resultado_v2["antecedentes"]) > 0

print("\nPRUEBA SIM-005 SUPERADA")

In [ ]:
# EP2 - Generación de orientación técnica con Gemini

from langchain_core.messages import HumanMessage

def generar_orientacion_ftth(descripcion, antecedentes):

    contexto = "\n\n".join(antecedentes)

    prompt = f"""
Eres un asistente técnico de apoyo para supervisores
de mantenimiento de redes de fibra óptica FTTH.

Tu tarea es analizar una incidencia utilizando
exclusivamente los antecedentes históricos proporcionados.

REGLAS OBLIGATORIAS:
1. No inventes mediciones, causas ni reparaciones.
2. No afirmes que un caso histórico demuestra la causa actual.
3. Diferencia los antecedentes de las verificaciones sugeridas.
4. Cita los identificadores FTTH de los casos utilizados.
5. Si falta información, indícalo claramente.
6. La decisión final corresponde al personal técnico.

INCIDENCIA ACTUAL:
{descripcion}

ANTECEDENTES RECUPERADOS:
{contexto}

Entrega una respuesta organizada en:

1. Resumen de la incidencia.
2. Antecedentes históricos relacionados.
3. Verificaciones técnicas sugeridas.
4. Información pendiente.
5. Advertencia sobre validación en terreno.

Utiliza un lenguaje técnico, claro y prudente.
"""

    respuesta = llm.invoke([
        HumanMessage(content=prompt)
    ])

    contenido = respuesta.content

    if isinstance(contenido, str):
        return contenido

    if isinstance(contenido, list):
        return "\n".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict)
            and bloque.get("type") == "text"
        )

    return str(contenido)


print("Función de orientación técnica creada correctamente")

In [ ]:
# EP2 - Prueba de orientación técnica con Gemini

print("=== GENERACIÓN DE ORIENTACIÓN FTTH ===")

orientacion_sim005 = generar_orientacion_ftth(
    descripcion=incidencia_sin_otdr["descripcion"],
    antecedentes=resultado_v2["antecedentes"]
)

print("\n=== RESPUESTA GENERADA POR GEMINI ===\n")
print(orientacion_sim005)

In [ ]:
# EP2 - Nodo de generación técnica con Gemini

def nodo_generar_orientacion(estado: EstadoFTTH):

    print("=== NODO GENERACIÓN GEMINI ===")

    descripcion = estado.get("descripcion", "")
    antecedentes = estado.get("antecedentes", [])

    if not antecedentes:
        return {
            "respuesta_final": (
                "No se dispone de antecedentes históricos "
                "para generar una orientación fundamentada."
            )
        }

    orientacion = generar_orientacion_ftth(
        descripcion=descripcion,
        antecedentes=antecedentes
    )

    print("Orientación técnica generada correctamente")

    return {
        "respuesta_final": orientacion
    }


# Construcción del grafo con RAG y Gemini

builder_v3 = StateGraph(EstadoFTTH)

builder_v3.add_node("supervisor", nodo_supervisor)
builder_v3.add_node(
    "solicitar_informacion",
    nodo_solicitar_informacion
)
builder_v3.add_node("consultar_rag", nodo_consultar_rag)
builder_v3.add_node(
    "generar_orientacion",
    nodo_generar_orientacion
)

builder_v3.add_edge(START, "supervisor")

builder_v3.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "consultar_rag": "consultar_rag"
    }
)

builder_v3.add_edge("solicitar_informacion", END)

builder_v3.add_edge(
    "consultar_rag",
    "generar_orientacion"
)

builder_v3.add_edge("generar_orientacion", END)

grafo_ftth_v3 = builder_v3.compile()

print("Grafo V3 con RAG y Gemini integrado correctamente")

In [ ]:
# EP2 - Prueba completa LangGraph + Chroma + Gemini

print("=== PRUEBA INTEGRADA AGENTE FTTH V3 ===")

resultado_v3 = grafo_ftth_v3.invoke(
    incidencia_sin_otdr
)

print("\n=== RESULTADO FINAL ===")
print("Ticket:", resultado_v3["ticket"])
print("Ruta seleccionada:", resultado_v3["siguiente_paso"])

print("\n=== ANTECEDENTES RECUPERADOS ===")
print("Cantidad:", len(resultado_v3["antecedentes"]))

print("\n=== ORIENTACIÓN TÉCNICA DEL AGENTE ===\n")
print(resultado_v3["respuesta_final"])

# Validaciones básicas
assert resultado_v3["siguiente_paso"] == "consultar_rag"
assert len(resultado_v3["antecedentes"]) > 0
assert len(resultado_v3["respuesta_final"].strip()) > 0

print("\nPRUEBA INTEGRADA V3 SUPERADA")

### 4.6 Integración completa de LangGraph, RAG y Gemini

Se realizó una prueba funcional del agente FTTH utilizando la incidencia simulada SIM-005, correspondiente a clientes sin servicio y un posible daño físico en una caja CTO.

El supervisor seleccionó la ruta de consulta RAG aun cuando no se disponía de una medición OTDR.

La búsqueda semántica recuperó tres antecedentes históricos de la base vectorial Chroma:

- FTTH-0306
- FTTH-0084
- FTTH-0405

Posteriormente, el nodo de generación ejecutó Gemini utilizando los antecedentes recuperados como contexto.

El sistema produjo una orientación técnica estructurada que incluyó:

1. Resumen de la incidencia.
2. Antecedentes históricos relacionados.
3. Verificaciones técnicas sugeridas.
4. Información pendiente para el diagnóstico.
5. Advertencia sobre validación en terreno.

**Resultado:** La prueba integrada V3 finalizó correctamente, confirmando el funcionamiento conjunto de LangGraph, Chroma y Gemini.

**Limitaciones actuales:** La selección de rutas todavía utiliza reglas condicionales. No se ha implementado memoria entre interacciones ni planificación adaptativa. La exactitud de los detalles históricos generados requiere validación adicional.

In [ ]:
# EP2 - Grafo FTTH V4 con memoria de corto plazo

from langgraph.checkpoint.memory import InMemorySaver

# Crear almacenamiento temporal de memoria
memoria_ftth = InMemorySaver()

# Compilar el grafo V3 incorporando memoria
grafo_ftth_v4 = builder_v3.compile(
    checkpointer=memoria_ftth
)

print("Grafo V4 con memoria de corto plazo creado correctamente")

In [ ]:
# EP2 - Prueba de memoria de corto plazo con LangGraph

configuracion_memoria = {
    "configurable": {
        "thread_id": "FTTH-SIM-005"
    }
}

# Ejecutar el agente con memoria
resultado_memoria = grafo_ftth_v4.invoke(
    incidencia_sin_otdr,
    config=configuracion_memoria
)

# Consultar el estado almacenado
estado_guardado = grafo_ftth_v4.get_state(
    configuracion_memoria
)

print("\n=== VERIFICACIÓN DE MEMORIA FTTH ===")
print("Ticket guardado:", estado_guardado.values["ticket"])
print("Descripción:", estado_guardado.values["descripcion"])
print("Distancia OTDR:", estado_guardado.values["distancia_otdr"])
print(
    "Antecedentes almacenados:",
    len(estado_guardado.values["antecedentes"])
)

print(
    "Respuesta técnica almacenada:",
    bool(estado_guardado.values["respuesta_final"])
)

assert estado_guardado.values["ticket"] == "SIM-005"
assert len(estado_guardado.values["antecedentes"]) == 3
assert estado_guardado.values["respuesta_final"]

print("\nPRUEBA DE MEMORIA FTTH SUPERADA")

In [ ]:
# EP2 - Actualización del estado de una incidencia

print("=== ACTUALIZACIÓN DE TICKET FTTH ===")

# Agregar una medición posterior al mismo ticket
grafo_ftth_v4.update_state(
    configuracion_memoria,
    {
        "distancia_otdr": 850.0
    }
)

# Recuperar el estado actualizado
estado_actualizado = grafo_ftth_v4.get_state(
    configuracion_memoria
)

datos = estado_actualizado.values

print("\n=== ESTADO ACTUALIZADO ===")
print("Ticket:", datos["ticket"])
print("Descripción:", datos["descripcion"])
print("Distancia OTDR:", datos["distancia_otdr"], "metros")
print("Antecedentes conservados:", len(datos["antecedentes"]))
print("Orientación anterior conservada:", bool(datos["respuesta_final"]))

# Validaciones
assert datos["ticket"] == "SIM-005"
assert datos["distancia_otdr"] == 850.0
assert len(datos["antecedentes"]) == 3
assert datos["respuesta_final"]

print("\nPRUEBA DE ACTUALIZACIÓN DE MEMORIA SUPERADA")

### 4.7 Implementación y validación de memoria de corto plazo

Se incorporó memoria de corto plazo al agente FTTH utilizando `InMemorySaver` de LangGraph.

Esta implementación permite almacenar el estado de una incidencia mediante un identificador de conversación (`thread_id`), conservando su descripción, mediciones, antecedentes históricos y orientación técnica generada.

**Prueba de almacenamiento:**

Se utilizó la incidencia simulada SIM-005, correspondiente a clientes FTTH sin servicio y posible daño físico en una CTO.

El sistema almacenó correctamente:

- Identificador del ticket.
- Descripción original de la incidencia.
- Tres antecedentes históricos recuperados mediante RAG.
- Orientación técnica generada por Gemini.
- Ausencia inicial de medición OTDR.

**Prueba de actualización:**

Posteriormente, se incorporó una distancia OTDR simulada de 850 metros al mismo ticket mediante `update_state()`.

Se comprobó que la nueva medición quedó registrada sin perder la descripción, los antecedentes ni la orientación anterior.

**Resultado:** Ambas pruebas finalizaron correctamente.

**Limitaciones:** La memoria permanece en RAM y se pierde al reiniciar el entorno de ejecución. Además, la actualización del estado no genera automáticamente una nueva evaluación técnica; esta capacidad se desarrollará posteriormente.

In [ ]:
# EP2 - Herramienta de planificación técnica FTTH

def herramienta_planificacion_ftth(estado: EstadoFTTH):

    print("=== PLANIFICACIÓN TÉCNICA FTTH ===")

    descripcion = estado.get("descripcion", "").lower()
    distancia = estado.get("distancia_otdr")

    plan = []

    # Verificaciones generales
    plan.append(
        "Confirmar identificación del nodo, CTO/NAP "
        "y cantidad de clientes afectados."
    )

    # Verificaciones según descripción
    if "cto" in descripcion or "nap" in descripcion:
        plan.append(
            "Inspeccionar físicamente la CTO/NAP, "
            "sus conectores, puertos y cables drop."
        )

    if "sin servicio" in descripcion:
        plan.append(
            "Medir potencia óptica y determinar si "
            "la afectación corresponde a clientes "
            "individuales o a una rama PON."
        )

    if "atenuación" in descripcion or "degradación" in descripcion:
        plan.append(
            "Comparar potencias ópticas entre puntos "
            "de la ruta para identificar pérdidas anormales."
        )

    # Verificaciones según disponibilidad OTDR
    if distancia is None:
        plan.append(
            "Solicitar medición OTDR si las verificaciones "
            "iniciales justifican su utilización."
        )
    else:
        plan.append(
            f"Analizar el evento OTDR reportado a {distancia} metros, "
            "confirmando origen de medición, sentido de la traza, "
            "longitud de onda y características del evento."
        )

        plan.append(
            "Contrastar la distancia OTDR con los planos "
            "y la topología de red antes de definir "
            "un punto de intervención."
        )

    plan.append(
        "Validar las conclusiones mediante mediciones "
        "y revisión del personal técnico en terreno."
    )

    print("Verificaciones propuestas:", len(plan))

    return {"plan_revision": plan}


print("Herramienta de planificación FTTH creada correctamente")

In [ ]:
# EP2 - Prueba de planificación con estado almacenado

print("=== PRUEBA DE PLANIFICACIÓN FTTH ===")

# Recuperar información actualizada desde la memoria
estado_actual = grafo_ftth_v4.get_state(
    configuracion_memoria
).values

# Generar plan utilizando los datos del ticket
resultado_plan = herramienta_planificacion_ftth(
    estado_actual
)

print("\n=== PLAN DE REVISIÓN PROPUESTO ===")

for numero, actividad in enumerate(
    resultado_plan["plan_revision"], start=1
):
    print(f"{numero}. {actividad}")

# Validaciones
assert len(resultado_plan["plan_revision"]) > 0

assert any(
    "850" in actividad
    for actividad in resultado_plan["plan_revision"]
)

print("\nPRUEBA DE PLANIFICACIÓN FTTH SUPERADA")

In [ ]:
# EP2 - Grafo V5 con planificación y memoria

builder_v5 = StateGraph(EstadoFTTH)

builder_v5.add_node("supervisor", nodo_supervisor)
builder_v5.add_node(
    "solicitar_informacion",
    nodo_solicitar_informacion
)
builder_v5.add_node("consultar_rag", nodo_consultar_rag)
builder_v5.add_node(
    "planificar_revision",
    herramienta_planificacion_ftth
)
builder_v5.add_node(
    "generar_orientacion",
    nodo_generar_orientacion
)

builder_v5.add_edge(START, "supervisor")

builder_v5.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "consultar_rag": "consultar_rag"
    }
)

builder_v5.add_edge("solicitar_informacion", END)

builder_v5.add_edge(
    "consultar_rag",
    "planificar_revision"
)

builder_v5.add_edge(
    "planificar_revision",
    "generar_orientacion"
)

builder_v5.add_edge("generar_orientacion", END)

# Memoria independiente para la versión V5
memoria_ftth_v5 = InMemorySaver()

grafo_ftth_v5 = builder_v5.compile(
    checkpointer=memoria_ftth_v5
)

print("Grafo V5 con RAG, planificación, Gemini y memoria creado correctamente")

In [ ]:
# EP2 - Prueba integrada del agente FTTH V5

print("=== PRUEBA INTEGRADA AGENTE FTTH V5 ===")

# Recuperar los datos actualizados del ticket
incidencia_v5 = dict(
    grafo_ftth_v4.get_state(
        configuracion_memoria
    ).values
)

# Configuración de memoria independiente para V5
configuracion_v5 = {
    "configurable": {
        "thread_id": "FTTH-SIM-005-V5"
    }
}

# Ejecutar el grafo completo
resultado_v5 = grafo_ftth_v5.invoke(
    incidencia_v5,
    config=configuracion_v5
)

print("\n=== RESULTADO GENERAL ===")
print("Ticket:", resultado_v5["ticket"])
print("Ruta:", resultado_v5["siguiente_paso"])
print("Distancia OTDR:", resultado_v5["distancia_otdr"])
print("Antecedentes RAG:", len(resultado_v5["antecedentes"]))

print("\n=== PLAN DE REVISIÓN ===")

for numero, actividad in enumerate(
    resultado_v5["plan_revision"], start=1
):
    print(f"{numero}. {actividad}")

print("\n=== ORIENTACIÓN GEMINI ===")
print(resultado_v5["respuesta_final"][:1000])

# Comprobar memoria V5
estado_memoria_v5 = grafo_ftth_v5.get_state(
    configuracion_v5
).values

assert estado_memoria_v5["ticket"] == "SIM-005"
assert estado_memoria_v5["distancia_otdr"] == 850.0
assert len(estado_memoria_v5["antecedentes"]) == 3
assert len(estado_memoria_v5["plan_revision"]) == 6
assert estado_memoria_v5["respuesta_final"].strip()

print("\nPRUEBA INTEGRADA V5 SUPERADA")

In [ ]:
# EP2 - Generación contextual con RAG, planificación y mediciones

def nodo_generar_orientacion_v2(estado: EstadoFTTH):

    print("=== GEMINI CON CONTEXTO COMPLETO ===")

    descripcion = estado.get("descripcion", "")
    antecedentes = estado.get("antecedentes", [])
    plan = estado.get("plan_revision", [])
    distancia = estado.get("distancia_otdr")

    if not antecedentes:
        return {
            "respuesta_final": (
                "No existen antecedentes suficientes "
                "para generar una orientación fundamentada."
            )
        }

    contexto_rag = "\n\n".join(antecedentes)

    contexto_plan = "\n".join(
        f"{i}. {actividad}"
        for i, actividad in enumerate(plan, start=1)
    )

    distancia_texto = (
        f"{distancia} metros"
        if distancia is not None
        else "No disponible"
    )

    prompt = f"""
Eres un asistente técnico de apoyo para supervisores
de mantenimiento de redes FTTH.

INCIDENCIA ACTUAL:
{descripcion}

DISTANCIA OTDR REPORTADA:
{distancia_texto}

ANTECEDENTES HISTÓRICOS:
{contexto_rag}

PLAN PRELIMINAR DE VERIFICACIÓN:
{contexto_plan}

INSTRUCCIONES:
1. Utiliza los antecedentes históricos como referencia.
2. Cita los identificadores FTTH correspondientes.
3. Considera el plan preliminar de verificaciones.
4. Diferencia datos actuales de antecedentes históricos.
5. No inventes mediciones, causas ni reparaciones.
6. No asumas que la distancia OTDR identifica la falla.
7. Solicita origen, sentido y parámetros de medición
   cuando no estén disponibles.
8. No recomiendes intervenir un elemento específico
   sin evidencia suficiente.
9. La decisión final corresponde al personal técnico.

Organiza la respuesta en:

1. Resumen de la incidencia actual.
2. Antecedentes históricos relevantes.
3. Plan de verificaciones recomendado.
4. Información pendiente.
5. Advertencia sobre validación en terreno.

Utiliza lenguaje técnico, claro y prudente.
"""

    respuesta = llm.invoke(prompt)

    contenido = respuesta.content

    if isinstance(contenido, str):
        texto = contenido
    elif isinstance(contenido, list):
        texto = "\n".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict)
            and bloque.get("type") == "text"
        )
    else:
        texto = str(contenido)

    print("Orientación contextual generada correctamente")

    return {"respuesta_final": texto}


print("Nodo Gemini V2 con contexto completo creado correctamente")

In [ ]:
# EP2 - Grafo V6 con generación contextual y memoria

builder_v6 = StateGraph(EstadoFTTH)

builder_v6.add_node("supervisor", nodo_supervisor)

builder_v6.add_node(
    "solicitar_informacion",
    nodo_solicitar_informacion
)

builder_v6.add_node(
    "consultar_rag",
    nodo_consultar_rag
)

builder_v6.add_node(
    "planificar_revision",
    herramienta_planificacion_ftth
)

builder_v6.add_node(
    "generar_orientacion",
    nodo_generar_orientacion_v2
)

builder_v6.add_edge(START, "supervisor")

builder_v6.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "consultar_rag": "consultar_rag"
    }
)

builder_v6.add_edge(
    "solicitar_informacion",
    END
)

builder_v6.add_edge(
    "consultar_rag",
    "planificar_revision"
)

builder_v6.add_edge(
    "planificar_revision",
    "generar_orientacion"
)

builder_v6.add_edge(
    "generar_orientacion",
    END
)

# Memoria independiente para V6
memoria_ftth_v6 = InMemorySaver()

grafo_ftth_v6 = builder_v6.compile(
    checkpointer=memoria_ftth_v6
)

print(
    "Grafo V6 con RAG, planificación contextual, "
    "Gemini V2 y memoria creado correctamente"
)

In [ ]:
# EP2 - Prueba integrada del agente FTTH V6

print("=== PRUEBA INTEGRADA AGENTE FTTH V6 ===")

configuracion_v6 = {
    "configurable": {
        "thread_id": "FTTH-SIM-005-V6"
    }
}

# Preparar la incidencia con la medición actualizada
incidencia_v6 = dict(incidencia_sin_otdr)
incidencia_v6["distancia_otdr"] = 850.0

# Ejecutar el agente completo
resultado_v6 = grafo_ftth_v6.invoke(
    incidencia_v6,
    config=configuracion_v6
)

print("\n=== RESULTADO GENERAL ===")
print("Ticket:", resultado_v6["ticket"])
print("Ruta:", resultado_v6["siguiente_paso"])
print("Distancia OTDR:", resultado_v6["distancia_otdr"])
print("Antecedentes RAG:", len(resultado_v6["antecedentes"]))
print("Verificaciones:", len(resultado_v6["plan_revision"]))

print("\n=== ORIENTACIÓN CONTEXTUAL GEMINI ===\n")
print(resultado_v6["respuesta_final"])

# Verificar memoria
estado_v6 = grafo_ftth_v6.get_state(
    configuracion_v6
).values

assert estado_v6["ticket"] == "SIM-005"
assert estado_v6["distancia_otdr"] == 850.0
assert len(estado_v6["antecedentes"]) == 3
assert len(estado_v6["plan_revision"]) == 6
assert estado_v6["respuesta_final"].strip()

print("\nPRUEBA INTEGRADA V6 SUPERADA")

In [ ]:
# EP2 - Actualización controlada de una incidencia FTTH

def actualizar_medicion_ftth(
    grafo,
    configuracion,
    nueva_distancia
):
    print("=== ACTUALIZACIÓN DE MEDICIÓN FTTH ===")

    estado_anterior = grafo.get_state(configuracion)

    if not estado_anterior.values:
        raise ValueError(
            "No existe un estado previo para este ticket."
        )

    ticket = estado_anterior.values.get("ticket")

    print("Ticket:", ticket)
    print(
        "Distancia anterior:",
        estado_anterior.values.get("distancia_otdr")
    )

    grafo.update_state(
        configuracion,
        {"distancia_otdr": nueva_distancia}
    )

    estado_actualizado = grafo.get_state(configuracion)

    print(
        "Nueva distancia:",
        estado_actualizado.values.get("distancia_otdr")
    )

    print("Medición actualizada en memoria")

    return estado_actualizado.values


print("Función de actualización FTTH creada correctamente")

In [ ]:
# EP2 - Prueba de actualización de medición OTDR

print("=== PRUEBA DE ACTUALIZACIÓN SIM-005 ===")

# Actualizar la distancia OTDR
estado_actualizado_v6 = actualizar_medicion_ftth(
    grafo_ftth_v6,
    configuracion_v6,
    1250.0
)

# Comprobar que la información se conserva
print("\n=== ESTADO ACTUALIZADO ===")

print("Ticket:", estado_actualizado_v6["ticket"])
print("Distancia OTDR:", estado_actualizado_v6["distancia_otdr"])
print("Antecedentes RAG:", len(estado_actualizado_v6["antecedentes"]))
print("Plan de revisión:", len(estado_actualizado_v6["plan_revision"]))

# Validaciones
assert estado_actualizado_v6["ticket"] == "SIM-005"
assert estado_actualizado_v6["distancia_otdr"] == 1250.0
assert len(estado_actualizado_v6["antecedentes"]) == 3
assert len(estado_actualizado_v6["plan_revision"]) == 6

print("\nPRUEBA DE ACTUALIZACIÓN SUPERADA")

In [ ]:
# EP2 - Reevaluación contextual de incidencia FTTH

print("=== REEVALUACIÓN TÉCNICA SIM-005 ===")

# 1. Recuperar el estado actualizado
estado_previo = dict(
    grafo_ftth_v6.get_state(
        configuracion_v6
    ).values
)

# 2. Recalcular el plan con la nueva distancia
resultado_nuevo_plan = herramienta_planificacion_ftth(
    estado_previo
)

estado_previo.update(resultado_nuevo_plan)

# 3. Generar una nueva orientación contextual
resultado_nueva_orientacion = nodo_generar_orientacion_v2(
    estado_previo
)

# 4. Guardar los resultados actualizados
grafo_ftth_v6.update_state(
    configuracion_v6,
    {
        "plan_revision": resultado_nuevo_plan["plan_revision"],
        "respuesta_final": resultado_nueva_orientacion["respuesta_final"]
    }
)

# 5. Verificar el estado final
estado_reevaluado = grafo_ftth_v6.get_state(
    configuracion_v6
).values

print("\n=== NUEVO PLAN DE REVISIÓN ===")

for numero, actividad in enumerate(
    estado_reevaluado["plan_revision"], start=1
):
    print(f"{numero}. {actividad}")

print("\n=== NUEVA ORIENTACIÓN GEMINI ===\n")
print(estado_reevaluado["respuesta_final"])

# Validaciones
assert estado_reevaluado["distancia_otdr"] == 1250.0
assert len(estado_reevaluado["antecedentes"]) == 3
assert any(
    "1250" in actividad
    for actividad in estado_reevaluado["plan_revision"]
)
assert estado_reevaluado["respuesta_final"].strip()

print("\nREEVALUACIÓN CONTEXTUAL SUPERADA")

In [ ]:
# EP2 - Historial de mediciones OTDR por incidencia

from datetime import datetime, timezone

historial_mediciones_ftth = {}

def registrar_medicion_ftth(
    ticket,
    distancia,
    origen="No informado",
    sentido="No informado",
    longitud_onda="No informada",
    observacion=""
):
    if distancia is None or distancia < 0:
        raise ValueError(
            "La distancia OTDR debe ser un valor válido."
        )

    registro = {
        "fecha_registro": datetime.now(
            timezone.utc
        ).isoformat(),
        "ticket": ticket,
        "distancia_m": float(distancia),
        "origen": origen,
        "sentido": sentido,
        "longitud_onda": longitud_onda,
        "observacion": observacion
    }

    historial_mediciones_ftth.setdefault(
        ticket, []
    ).append(registro)

    print("=== MEDICIÓN OTDR REGISTRADA ===")
    print("Ticket:", ticket)
    print("Distancia:", distancia, "metros")
    print("Origen:", origen)
    print("Sentido:", sentido)
    print(
        "Total de mediciones:",
        len(historial_mediciones_ftth[ticket])
    )

    return registro


print("Herramienta de historial OTDR creada correctamente")

In [ ]:
# EP2 - Prueba de historial de mediciones OTDR

print("=== PRUEBA HISTORIAL OTDR SIM-005 ===")

# Primera medición simulada
registrar_medicion_ftth(
    ticket="SIM-005",
    distancia=850.0,
    origen="Mufa de distribución (simulada)",
    sentido="Hacia CTO",
    longitud_onda="1550 nm",
    observacion="Primera medición de referencia"
)

# Segunda medición simulada
registrar_medicion_ftth(
    ticket="SIM-005",
    distancia=1250.0,
    origen="Nodo FTTH (simulado)",
    sentido="Hacia red de distribución",
    longitud_onda="1550 nm",
    observacion="Segunda medición desde otro punto"
)

# Consultar el historial completo
historial_sim005 = historial_mediciones_ftth["SIM-005"]

print("\n=== HISTORIAL DEL TICKET SIM-005 ===")

for numero, medicion in enumerate(
    historial_sim005, start=1
):
    print(f"\nMedición {numero}")
    print("Distancia:", medicion["distancia_m"], "metros")
    print("Origen:", medicion["origen"])
    print("Sentido:", medicion["sentido"])
    print("Longitud de onda:", medicion["longitud_onda"])

# Validaciones
assert len(historial_sim005) == 2
assert historial_sim005[0]["distancia_m"] == 850.0
assert historial_sim005[1]["distancia_m"] == 1250.0

print("\nPRUEBA DE HISTORIAL OTDR SUPERADA")

In [ ]:
# EP2 - Ampliación del estado FTTH con historial OTDR

from typing import List, Dict, Any

class EstadoFTTHV2(EstadoFTTH):
    historial_mediciones: List[Dict[str, Any]]


print("=== ESTADO FTTH V2 ===")

print(
    "Campo historial_mediciones incorporado:",
    "historial_mediciones" in EstadoFTTHV2.__annotations__
)

print(
    "Total de campos del estado:",
    len(EstadoFTTHV2.__annotations__)
)

assert "historial_mediciones" in EstadoFTTHV2.__annotations__

print("\nESTADO FTTH V2 CREADO CORRECTAMENTE")

In [ ]:
# EP2 - Integración de historial OTDR al estado FTTH V2

from copy import deepcopy

print("=== INTEGRACIÓN HISTORIAL OTDR ===")

# Recuperar el estado actual del ticket SIM-005
estado_base = dict(
    grafo_ftth_v6.get_state(
        configuracion_v6
    ).values
)

# Recuperar las mediciones registradas
historial_ticket = deepcopy(
    historial_mediciones_ftth.get("SIM-005", [])
)

# Crear el estado ampliado
estado_ftth_v2 = {
    **estado_base,
    "historial_mediciones": historial_ticket
}

# Mostrar resultados
print("Ticket:", estado_ftth_v2["ticket"])
print("Distancia actual:", estado_ftth_v2["distancia_otdr"])
print("Antecedentes RAG:", len(estado_ftth_v2["antecedentes"]))
print("Plan de revisión:", len(estado_ftth_v2["plan_revision"]))
print(
    "Mediciones históricas:",
    len(estado_ftth_v2["historial_mediciones"])
)

print("\n=== MEDICIONES CONSERVADAS ===")

for numero, medicion in enumerate(
    estado_ftth_v2["historial_mediciones"], start=1
):
    print(
        f"{numero}. {medicion['distancia_m']} metros"
        f" | Origen: {medicion['origen']}"
        f" | Sentido: {medicion['sentido']}"
    )

# Validaciones
assert estado_ftth_v2["ticket"] == "SIM-005"
assert estado_ftth_v2["distancia_otdr"] == 1250.0
assert len(estado_ftth_v2["historial_mediciones"]) == 2
assert len(estado_ftth_v2["antecedentes"]) == 3
assert len(estado_ftth_v2["plan_revision"]) == 6

print("\nINTEGRACIÓN DE DATOS SUPERADA")

In [ ]:
# EP2 - Grafo V7 con historial OTDR integrado al estado

builder_v7 = StateGraph(EstadoFTTHV2)

builder_v7.add_node("supervisor", nodo_supervisor)

builder_v7.add_node(
    "solicitar_informacion",
    nodo_solicitar_informacion
)

builder_v7.add_node(
    "consultar_rag",
    nodo_consultar_rag
)

builder_v7.add_node(
    "planificar_revision",
    herramienta_planificacion_ftth
)

builder_v7.add_node(
    "generar_orientacion",
    nodo_generar_orientacion_v2
)

builder_v7.add_edge(START, "supervisor")

builder_v7.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "consultar_rag": "consultar_rag"
    }
)

builder_v7.add_edge(
    "solicitar_informacion",
    END
)

builder_v7.add_edge(
    "consultar_rag",
    "planificar_revision"
)

builder_v7.add_edge(
    "planificar_revision",
    "generar_orientacion"
)

builder_v7.add_edge(
    "generar_orientacion",
    END
)

# Memoria independiente para V7
memoria_ftth_v7 = InMemorySaver()

grafo_ftth_v7 = builder_v7.compile(
    checkpointer=memoria_ftth_v7
)

print(
    "Grafo V7 con historial OTDR, RAG, "
    "planificación, Gemini y memoria creado correctamente"
)

In [ ]:
# EP2 - Validación de historial OTDR en LangGraph V7

print("=== PRUEBA DE MEMORIA FTTH V7 ===")

# Configuración independiente para V7
configuracion_v7 = {
    "configurable": {
        "thread_id": "FTTH-SIM-005-V7"
    }
}

# Cargar el estado preparado en la memoria del grafo
grafo_ftth_v7.update_state(
    configuracion_v7,
    estado_ftth_v2,
    as_node="generar_orientacion"
)

# Recuperar el estado almacenado
estado_memoria_v7 = grafo_ftth_v7.get_state(
    configuracion_v7
).values

print("\n=== DATOS DEL TICKET ===")
print("Ticket:", estado_memoria_v7["ticket"])
print("Distancia actual:", estado_memoria_v7["distancia_otdr"])
print("Antecedentes RAG:", len(estado_memoria_v7["antecedentes"]))
print("Verificaciones:", len(estado_memoria_v7["plan_revision"]))

print("\n=== HISTORIAL OTDR EN LANGGRAPH ===")

for numero, medicion in enumerate(
    estado_memoria_v7["historial_mediciones"],
    start=1
):
    print(f"\nMedición {numero}")
    print("Distancia:", medicion["distancia_m"], "metros")
    print("Origen:", medicion["origen"])
    print("Sentido:", medicion["sentido"])
    print("Longitud de onda:", medicion["longitud_onda"])

# Validaciones
assert estado_memoria_v7["ticket"] == "SIM-005"
assert estado_memoria_v7["distancia_otdr"] == 1250.0
assert len(estado_memoria_v7["historial_mediciones"]) == 2
assert len(estado_memoria_v7["antecedentes"]) == 3
assert len(estado_memoria_v7["plan_revision"]) == 6

assert (
    estado_memoria_v7["historial_mediciones"][0]["distancia_m"]
    == 850.0
)

assert (
    estado_memoria_v7["historial_mediciones"][1]["distancia_m"]
    == 1250.0
)

print("\nPRUEBA DE MEMORIA V7 SUPERADA")

In [ ]:
# EP2 - Preparación del historial OTDR para Gemini

def preparar_contexto_otdr(estado):
    historial = estado.get("historial_mediciones", [])

    if not historial:
        return "No existen mediciones OTDR históricas registradas."

    lineas = []

    for numero, medicion in enumerate(historial, start=1):

        lineas.append(
            f"Medición {numero}:\n"
            f"- Distancia: {medicion.get('distancia_m')} metros\n"
            f"- Origen: {medicion.get('origen', 'No informado')}\n"
            f"- Sentido: {medicion.get('sentido', 'No informado')}\n"
            f"- Longitud de onda: "
            f"{medicion.get('longitud_onda', 'No informada')}\n"
            f"- Observación: "
            f"{medicion.get('observacion', 'Sin observación')}"
        )

    return "\n\n".join(lineas)


print("=== CONTEXTO HISTÓRICO OTDR ===")

contexto_otdr = preparar_contexto_otdr(
    estado_memoria_v7
)

print(contexto_otdr)

assert "850.0 metros" in contexto_otdr
assert "1250.0 metros" in contexto_otdr
assert "Mufa de distribución" in contexto_otdr
assert "Nodo FTTH" in contexto_otdr

print("\nPREPARACIÓN DE CONTEXTO OTDR SUPERADA")

In [ ]:
# EP2 - Gemini V3 con historial de mediciones OTDR

def nodo_generar_orientacion_v3(estado: EstadoFTTHV2):

    print("=== GEMINI V3 CON HISTORIAL OTDR ===")

    descripcion = estado.get("descripcion", "")
    antecedentes = estado.get("antecedentes", [])
    plan = estado.get("plan_revision", [])
    distancia = estado.get("distancia_otdr")

    if not antecedentes:
        return {
            "respuesta_final":
            "No existen antecedentes RAG suficientes "
            "para generar una orientación fundamentada."
        }

    contexto_rag = "\n\n".join(antecedentes)

    contexto_plan = "\n".join(
        f"{numero}. {actividad}"
        for numero, actividad in enumerate(plan, start=1)
    )

    contexto_historial = preparar_contexto_otdr(estado)

    distancia_actual = (
        f"{distancia} metros"
        if distancia is not None
        else "No disponible"
    )

    prompt = f"""
Eres un asistente técnico para supervisores de
mantenimiento de redes de fibra óptica FTTH.

INCIDENCIA ACTUAL:
{descripcion}

DISTANCIA OTDR ACTUAL:
{distancia_actual}

ANTECEDENTES HISTÓRICOS RAG:
{contexto_rag}

PLAN PRELIMINAR DE VERIFICACIÓN:
{contexto_plan}

HISTORIAL DE MEDICIONES OTDR:
{contexto_historial}

REGLAS OBLIGATORIAS:

1. No inventes mediciones, causas ni reparaciones.
2. Distingue los casos históricos de la incidencia actual.
3. Cita los identificadores FTTH de los antecedentes utilizados.
4. No atribuyas a los casos históricos datos que no estén
   expresamente presentes en sus documentos.
5. Considera el origen y sentido de cada medición OTDR.
6. No asumas que mediciones realizadas desde puntos
   distintos corresponden al mismo evento óptico.
7. No calcules diferencias de distancias como ubicación
   de una falla sin verificar la topología.
8. No afirmes que la distancia OTDR confirma una falla.
9. Identifica la información técnica que falta.
10. Diferencia un medidor de potencia óptica (OPM)
    de un localizador visual de fallas (VFL).
11. La decisión final corresponde al personal técnico.

ORGANIZA LA RESPUESTA EN:

1. Resumen de la incidencia.
2. Antecedentes históricos relevantes.
3. Análisis del historial OTDR.
4. Plan de verificaciones recomendado.
5. Información pendiente.
6. Advertencia de validación en terreno.

Utiliza lenguaje técnico, claro y prudente.
"""

    respuesta = llm.invoke(
        [HumanMessage(content=prompt)]
    )

    contenido = respuesta.content

    if isinstance(contenido, str):
        texto = contenido

    elif isinstance(contenido, list):
        texto = "\n".join(
            bloque.get("text", "")
            for bloque in contenido
            if isinstance(bloque, dict)
            and bloque.get("type") == "text"
        )

    else:
        texto = str(contenido)

    print("Orientación con historial OTDR generada")

    return {"respuesta_final": texto}


print("Nodo Gemini V3 con historial OTDR creado correctamente")

In [ ]:
# EP2 - Validación del contexto de mediciones OTDR

def validar_contexto_mediciones_otdr(estado):

    print("=== VALIDACIÓN DE CONTEXTO OTDR ===")

    historial = estado.get("historial_mediciones", [])

    if not historial:
        return {
            "estado": "sin_mediciones",
            "advertencia": "No existen mediciones registradas."
        }

    origenes = {
        str(m.get("origen", "")).strip().lower()
        for m in historial
    }

    sentidos = {
        str(m.get("sentido", "")).strip().lower()
        for m in historial
    }

    if len(origenes) > 1:
        advertencia = (
            "Existen mediciones desde diferentes puntos "
            "de origen. No se deben sumar ni restar "
            "distancias para localizar una falla sin "
            "confirmar que corresponden al mismo evento, "
            "fibra y recorrido óptico."
        )
        estado_validacion = "origenes_diferentes"

    else:
        advertencia = (
            "Las mediciones presentan el mismo origen "
            "registrado. Se requiere verificar los "
            "parámetros OTDR y la identidad del evento "
            "antes de compararlas."
        )
        estado_validacion = "mismo_origen"

    print("Mediciones:", len(historial))
    print("Orígenes distintos:", len(origenes))
    print("Sentidos distintos:", len(sentidos))
    print("Resultado:", estado_validacion)
    print("Advertencia:", advertencia)

    return {
        "estado": estado_validacion,
        "advertencia": advertencia
    }


# Probar con el historial de SIM-005
validacion_otdr = validar_contexto_mediciones_otdr(
    estado_memoria_v7
)

assert validacion_otdr["estado"] == "origenes_diferentes"

print("\nVALIDACIÓN DE CONTEXTO OTDR SUPERADA")

In [ ]:
# EP2 - Gemini V4 con validación técnica OTDR

def nodo_generar_orientacion_v4(estado: EstadoFTTHV2):

    print("=== GEMINI V4 CON VALIDACIÓN OTDR ===")

    # Validar las mediciones antes del análisis
    validacion = validar_contexto_mediciones_otdr(estado)

    # Generar orientación contextual con historial
    resultado = nodo_generar_orientacion_v3(estado)

    # Incorporar advertencia técnica al resultado
    advertencia = validacion["advertencia"]

    respuesta_final = (
        "VALIDACIÓN PREVIA DE MEDICIONES OTDR\n"
        f"Estado: {validacion['estado']}\n"
        f"Advertencia: {advertencia}\n\n"
        "ORIENTACIÓN TÉCNICA\n"
        f"{resultado['respuesta_final']}"
    )

    print("Validación OTDR incorporada a la orientación")

    return {
        "respuesta_final": respuesta_final
    }


print("Nodo Gemini V4 con validación OTDR creado correctamente")

In [ ]:
# EP2 - Grafo V8 con validación técnica OTDR

builder_v8 = StateGraph(EstadoFTTHV2)

builder_v8.add_node(
    "supervisor",
    nodo_supervisor
)

builder_v8.add_node(
    "solicitar_informacion",
    nodo_solicitar_informacion
)

builder_v8.add_node(
    "consultar_rag",
    nodo_consultar_rag
)

builder_v8.add_node(
    "planificar_revision",
    herramienta_planificacion_ftth
)

builder_v8.add_node(
    "generar_orientacion",
    nodo_generar_orientacion_v4
)

# Inicio del flujo
builder_v8.add_edge(START, "supervisor")

# Decisión del supervisor
builder_v8.add_conditional_edges(
    "supervisor",
    lambda estado: estado["siguiente_paso"],
    {
        "solicitar_informacion": "solicitar_informacion",
        "consultar_rag": "consultar_rag"
    }
)

# Rutas del agente
builder_v8.add_edge(
    "solicitar_informacion",
    END
)

builder_v8.add_edge(
    "consultar_rag",
    "planificar_revision"
)

builder_v8.add_edge(
    "planificar_revision",
    "generar_orientacion"
)

builder_v8.add_edge(
    "generar_orientacion",
    END
)

# Memoria independiente de V8
memoria_ftth_v8 = InMemorySaver()

grafo_ftth_v8 = builder_v8.compile(
    checkpointer=memoria_ftth_v8
)

print(
    "Grafo V8 con validación OTDR, historial, "
    "RAG, planificación, Gemini y memoria creado correctamente"
)

In [ ]:
# EP2 - Prueba integrada del agente FTTH V8

print("=== PRUEBA INTEGRADA AGENTE FTTH V8 ===")

# Configuración independiente de memoria
configuracion_v8 = {
    "configurable": {
        "thread_id": "FTTH-SIM-005-V8"
    }
}

# Preparar el ticket con historial OTDR
from copy import deepcopy

incidencia_v8 = deepcopy(estado_ftth_v2)

# Ejecutar el grafo completo
resultado_v8 = grafo_ftth_v8.invoke(
    incidencia_v8,
    config=configuracion_v8
)

print("\n=== RESULTADO GENERAL ===")
print("Ticket:", resultado_v8["ticket"])
print("Distancia actual:", resultado_v8["distancia_otdr"])
print("Antecedentes RAG:", len(resultado_v8["antecedentes"]))
print("Verificaciones:", len(resultado_v8["plan_revision"]))
print(
    "Mediciones históricas:",
    len(resultado_v8["historial_mediciones"])
)

print("\n=== ORIENTACIÓN TÉCNICA ===\n")
print(resultado_v8["respuesta_final"])

# Verificar el estado almacenado
estado_guardado_v8 = grafo_ftth_v8.get_state(
    configuracion_v8
).values

assert estado_guardado_v8["ticket"] == "SIM-005"
assert estado_guardado_v8["distancia_otdr"] == 1250.0
assert len(estado_guardado_v8["antecedentes"]) == 3
assert len(estado_guardado_v8["plan_revision"]) == 6
assert len(estado_guardado_v8["historial_mediciones"]) == 2

assert (
    "origenes_diferentes"
    in estado_guardado_v8["respuesta_final"]
)

print("\nPRUEBA INTEGRADA V8 SUPERADA")

In [ ]:
# EP2 - Exportación de evidencia técnica V8

import json
from pathlib import Path
from datetime import datetime, timezone

print("=== EXPORTACIÓN DE EVIDENCIA V8 ===")

# Recuperar el estado validado
estado_evidencia = dict(
    grafo_ftth_v8.get_state(
        configuracion_v8
    ).values
)

# Preparar evidencia
evidencia_v8 = {
    "proyecto": "EP2 Agente Inteligente FTTH",
    "version": "V8",
    "fecha_exportacion": datetime.now(
        timezone.utc
    ).isoformat(),
    "tipo_prueba": "Integración RAG, planificación, OTDR y Gemini",
    "datos_simulados": True,
    "estado_ticket": estado_evidencia,
    "validacion_otdr": validar_contexto_mediciones_otdr(
        estado_evidencia
    )
}

# Guardar archivo JSON
ruta_evidencia = Path(
    "/content/EP2_Evidencia_Prueba_V8_SIM005.json"
)

with open(ruta_evidencia, "w", encoding="utf-8") as archivo:
    json.dump(
        evidencia_v8,
        archivo,
        ensure_ascii=False,
        indent=4
    )

# Verificaciones
assert ruta_evidencia.exists()
assert evidencia_v8["estado_ticket"]["ticket"] == "SIM-005"
assert len(
    evidencia_v8["estado_ticket"]["historial_mediciones"]
) == 2

print("Archivo generado:", ruta_evidencia)
print("Tamaño:", ruta_evidencia.stat().st_size, "bytes")
print("\nEVIDENCIA V8 EXPORTADA CORRECTAMENTE")

In [ ]:
# EP2 - Paso 51: Gráfico de historial OTDR

import matplotlib.pyplot as plt
from pathlib import Path

print("=== GENERACIÓN DE GRÁFICO OTDR ===")

# Recuperar información desde LangGraph V8
estado_grafico = grafo_ftth_v8.get_state(
    configuracion_v8
).values

ticket = estado_grafico["ticket"]
mediciones = estado_grafico["historial_mediciones"]

# Preparar datos
etiquetas = [
    f"Medición {i}\n{m['origen']}"
    for i, m in enumerate(mediciones, start=1)
]

distancias = [
    m["distancia_m"]
    for m in mediciones
]

# Crear gráfico
fig, ax = plt.subplots(figsize=(10, 6))

barras = ax.bar(
    etiquetas,
    distancias,
    color=["#2563EB", "#16A34A"][:len(distancias)],
    width=0.5
)

# Mostrar valores sobre las barras
for barra, distancia in zip(barras, distancias):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 25,
        f"{distancia:,.0f} m",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

ax.set_title(
    f"Historial de mediciones OTDR - {ticket}",
    fontsize=15,
    fontweight="bold",
    pad=20
)

ax.set_ylabel("Distancia OTDR reportada (metros)")
ax.set_ylim(0, max(distancias) * 1.25)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

fig.text(
    0.5, 0.04,
    "Datos simulados. Distintos orígenes de medición.\n"
    "Las distancias no confirman una ubicación de falla.",
    ha="center",
    fontsize=10
)

plt.tight_layout(rect=[0, 0.11, 1, 1])

# Guardar evidencia
ruta_grafico = Path(
    "/content/EP2_Grafico_OTDR_SIM005.png"
)

plt.savefig(
    ruta_grafico,
    dpi=200,
    bbox_inches="tight"
)

plt.show()

# Validaciones
assert len(mediciones) == 2
assert ruta_grafico.exists()

print("\nGráfico guardado en:", ruta_grafico)
print("GRÁFICO OTDR GENERADO CORRECTAMENTE")

In [ ]:
# EP2 - Paso 52: Respaldo de evidencias en Google Drive

from google.colab import drive
from pathlib import Path
import shutil

print("=== RESPALDO DE EVIDENCIAS EP2 ===")

# Conectar Google Drive
drive.mount("/content/drive")

# Crear carpeta de evidencias
carpeta_destino = Path(
    "/content/drive/MyDrive/EP2_Agente_Inteligente_FTTH/Evidencias"
)

carpeta_destino.mkdir(
    parents=True,
    exist_ok=True
)

# Archivos generados anteriormente
archivos = [
    Path("/content/EP2_Evidencia_Prueba_V8_SIM005.json"),
    Path("/content/EP2_Grafico_OTDR_SIM005.png")
]

# Copiar archivos a Drive
for archivo in archivos:

    if not archivo.exists():
        raise FileNotFoundError(
            f"No se encontró: {archivo}"
        )

    destino = carpeta_destino / archivo.name

    shutil.copy2(archivo, destino)

    assert destino.exists()

    print("Respaldado:", destino.name)

print("\nCarpeta:", carpeta_destino)
print("RESPALDO DE EVIDENCIAS COMPLETADO")

In [ ]:
# EP2 - Paso 53: Verificación de evidencias en Drive

import json
from pathlib import Path

print("=== VERIFICACIÓN DE RESPALDOS EP2 ===")

carpeta = Path(
    "/content/drive/MyDrive/EP2_Agente_Inteligente_FTTH/Evidencias"
)

ruta_json = carpeta / "EP2_Evidencia_Prueba_V8_SIM005.json"
ruta_png = carpeta / "EP2_Grafico_OTDR_SIM005.png"

assert ruta_json.exists()
assert ruta_png.exists()

with open(ruta_json, "r", encoding="utf-8") as archivo:
    evidencia = json.load(archivo)

ticket = evidencia["estado_ticket"]["ticket"]
historial = evidencia["estado_ticket"]["historial_mediciones"]

print("Ticket:", ticket)
print("Mediciones OTDR:", len(historial))
print("Gráfico PNG:", ruta_png.name)
print("Tamaño PNG:", ruta_png.stat().st_size, "bytes")

assert ticket == "SIM-005"
assert len(historial) == 2
assert ruta_png.stat().st_size > 0

print("\nRESPALDOS EP2 VERIFICADOS CORRECTAMENTE")

In [ ]:
# EP2 - Paso 54: Persistencia del estado FTTH en Google Drive

import json
from pathlib import Path
from datetime import datetime, timezone

print("=== PERSISTENCIA DEL ESTADO FTTH ===")

# Carpeta para guardar estados de incidencias
carpeta_memoria = Path(
    "/content/drive/MyDrive/"
    "EP2_Agente_Inteligente_FTTH/Memoria"
)

carpeta_memoria.mkdir(
    parents=True,
    exist_ok=True
)

# Recuperar el estado actual desde LangGraph V8
estado_actual = dict(
    grafo_ftth_v8.get_state(
        configuracion_v8
    ).values
)

# Preparar el registro persistente
registro_memoria = {
    "proyecto": "EP2 Agente Inteligente FTTH",
    "version_agente": "V8",
    "fecha_guardado": datetime.now(
        timezone.utc
    ).isoformat(),
    "estado": estado_actual
}

# Identificar el archivo por ticket
ticket = estado_actual["ticket"]

ruta_memoria = carpeta_memoria / f"{ticket}_estado.json"

# Guardar en Google Drive
with open(ruta_memoria, "w", encoding="utf-8") as archivo:
    json.dump(
        registro_memoria,
        archivo,
        ensure_ascii=False,
        indent=4
    )

# Validaciones
assert ruta_memoria.exists()
assert ruta_memoria.stat().st_size > 0

print("Ticket:", ticket)
print("Archivo:", ruta_memoria.name)
print("Mediciones:", len(
    estado_actual["historial_mediciones"]
))
print("Antecedentes RAG:", len(
    estado_actual["antecedentes"]
))

print("\nESTADO FTTH GUARDADO EN DRIVE CORRECTAMENTE")

In [ ]:
# EP2 - Paso 55: Recuperación del estado desde Google Drive

import json
from pathlib import Path

print("=== RECUPERACIÓN DE MEMORIA FTTH ===")

# Archivo persistente
ruta_memoria = Path(
    "/content/drive/MyDrive/"
    "EP2_Agente_Inteligente_FTTH/Memoria/"
    "SIM-005_estado.json"
)

# Verificar existencia
assert ruta_memoria.exists(), "No se encontró el archivo de memoria"

# Recuperar información
with open(ruta_memoria, "r", encoding="utf-8") as archivo:
    memoria_recuperada = json.load(archivo)

estado_recuperado = memoria_recuperada["estado"]

# Mostrar información
print("Ticket:", estado_recuperado["ticket"])
print("Descripción:", estado_recuperado["descripcion"])
print("Distancia actual:", estado_recuperado["distancia_otdr"])
print("Antecedentes RAG:", len(estado_recuperado["antecedentes"]))
print("Verificaciones:", len(estado_recuperado["plan_revision"]))
print(
    "Mediciones históricas:",
    len(estado_recuperado["historial_mediciones"])
)

print("\n=== HISTORIAL OTDR RECUPERADO ===")

for numero, medicion in enumerate(
    estado_recuperado["historial_mediciones"], start=1
):
    print(
        f"{numero}. {medicion['distancia_m']} m"
        f" | Origen: {medicion['origen']}"
    )

# Validaciones
assert estado_recuperado["ticket"] == "SIM-005"
assert estado_recuperado["distancia_otdr"] == 1250.0
assert len(estado_recuperado["antecedentes"]) == 3
assert len(estado_recuperado["historial_mediciones"]) == 2
assert estado_recuperado["respuesta_final"].strip()

print("\nMEMORIA FTTH RECUPERADA CORRECTAMENTE")

In [ ]:
# EP2 - Paso 56: Restaurar memoria en LangGraph V8

from copy import deepcopy

print("=== RESTAURACIÓN DE MEMORIA EN LANGGRAPH ===")

# Crear una sesión independiente
configuracion_restaurada = {
    "configurable": {
        "thread_id": "FTTH-SIM-005-RESTAURADO"
    }
}

# Cargar una copia del estado recuperado desde Drive
grafo_ftth_v8.update_state(
    configuracion_restaurada,
    deepcopy(estado_recuperado),
    as_node="generar_orientacion"
)

# Consultar la nueva sesión
estado_restaurado = grafo_ftth_v8.get_state(
    configuracion_restaurada
).values

print("\n=== ESTADO RESTAURADO ===")

print("Ticket:", estado_restaurado["ticket"])
print("Distancia OTDR:", estado_restaurado["distancia_otdr"])
print("Antecedentes RAG:", len(estado_restaurado["antecedentes"]))
print("Verificaciones:", len(estado_restaurado["plan_revision"]))
print(
    "Mediciones históricas:",
    len(estado_restaurado["historial_mediciones"])
)

# Validar la recuperación
assert estado_restaurado["ticket"] == "SIM-005"
assert estado_restaurado["distancia_otdr"] == 1250.0
assert len(estado_restaurado["antecedentes"]) == 3
assert len(estado_restaurado["plan_revision"]) == 6
assert len(estado_restaurado["historial_mediciones"]) == 2
assert (
    estado_restaurado["respuesta_final"]
    == estado_recuperado["respuesta_final"]
)

print("\nRESTAURACIÓN EN LANGGRAPH SUPERADA")